# Course 4 - Pandas II: Cleaning, GroupBy, Joins, Time

Live-coding notebook that mirrors the slide deck.
Concrete examples lifted from Aurelien Geron's Pandas tutorial.

**Sections**
1. Missing data & column ops (0:00-0:30)
2. GroupBy, pivot, merge (0:30-1:00)
3. Time series (1:00-1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
import pandas as pd
import numpy as np
from shared.data_utils import load_dataset
rng = np.random.default_rng(0)


## 1. Missing data & column ops

### Load `titanic` - the canonical messy dataset

In [ ]:
titanic = load_dataset('titanic')
titanic.head()


### Find the NaNs

In [ ]:
titanic.isna().sum()


### Drop, fill, interpolate

In [ ]:
clean = titanic.dropna(subset=['embarked'])
print('rows lost:', len(titanic) - len(clean))


In [ ]:
filled = titanic.assign(age=titanic['age'].fillna(titanic['age'].median()))
print('age NaNs after fillna:', filled['age'].isna().sum())


### Interpolate across columns - Geron-style

In [ ]:
bonus = pd.DataFrame(
    [[0, np.nan, 2], [np.nan, 1, 0], [0, 1, 0], [3, 3, 0]],
    columns=['oct', 'nov', 'dec'],
    index=['bob', 'colin', 'darwin', 'charles'])
bonus.interpolate(axis=1)


### Column ops - `assign` and `query`

In [ ]:
tips = load_dataset('tips')
(tips
   .assign(tip_pct=lambda df: df['tip'] / df['total_bill'])
   .query('tip_pct > 0.2 and time == "Dinner"')
   .head())


## 2. GroupBy, pivot, merge

### `groupby` + aggregation - the workhorse

In [ ]:
tips.groupby('day', observed=False)['tip'].mean()


In [ ]:
tips.groupby(['day', 'sex'], observed=False)['tip'].agg(['mean', 'count'])


### Pivot table - Geron's `more_grades` shape

In [ ]:
more = pd.DataFrame({
    'name':   ['alice', 'alice', 'bob', 'bob', 'charles', 'charles'],
    'month':  ['sep',   'oct',   'sep', 'oct', 'sep',     'oct'],
    'grade':  [ 8,       8,       10,    9,    4,         8],
})
more


In [ ]:
pd.pivot_table(more, index='name', values='grade',
               columns='month', margins=True)


### SQL-style joins with `pd.merge`

In [ ]:
city_loc = pd.DataFrame(
    [['CA', 'San Francisco', 37.78, -122.42],
     ['NY', 'New York',      40.71,  -74.01],
     ['FL', 'Miami',         25.79,  -80.32]],
    columns=['state', 'city', 'lat', 'lng'])

city_pop = pd.DataFrame(
    [[ 808976, 'San Francisco', 'California'],
     [8363710, 'New York',      'New-York'],
     [2242193, 'Houston',       'Texas']],
    columns=['population', 'city', 'state'])

pd.merge(city_loc, city_pop, on='city')               # INNER


In [ ]:
pd.merge(city_loc, city_pop, on='city', how='outer')  # OUTER


In [ ]:
pd.merge(city_loc, city_pop, on='city', how='left')   # LEFT


### Concatenation - vertical and horizontal

In [ ]:
pd.concat([city_loc, city_pop], ignore_index=True)   # vertical


In [ ]:
pd.concat([city_loc.set_index('city'), city_pop.set_index('city')], axis=1)   # horizontal


## 3. Time series

### `pd.date_range` and a DatetimeIndex

In [ ]:
temps = [4.4, 5.1, 6.1, 6.2, 6.1, 6.1, 5.7, 5.2, 4.7, 4.1, 3.9, 3.5]
dates = pd.date_range('2016-10-29 17:30', periods=12, freq='h')
ts = pd.Series(temps, index=dates, name='Temperature')
ts.head()


### Resample - down and up

In [ ]:
ts.resample('2h').mean()       # downsample: aggregate


In [ ]:
ts.resample('15min').interpolate(method='cubic').head()   # upsample: interpolate


### Real CSV - `stock-prices` (Plotly's 2014 AAPL closing prices)

In [ ]:
sp = load_dataset('stock-prices')
print(sp.columns.tolist())
sp.head()


In [ ]:
# The Plotly file labels its columns AAPL_x (date) and AAPL_y (price).
DATE  = sp.columns[0]
PRICE = sp.columns[1]
sp[DATE] = pd.to_datetime(sp[DATE])
sp = sp.sort_values(DATE).set_index(DATE)
sp.head()


### Daily -> weekly mean - and a quick `.plot()`

In [ ]:
weekly = sp[PRICE].resample('W').mean()
ax = weekly.plot(title='AAPL - weekly mean closing price')
ax.set_ylabel('USD');


### Recap
* `isna`/`dropna`/`fillna`/`interpolate` cover 90% of cleaning.
* `groupby`/`pivot_table`/`merge` cover 90% of reshaping.
* `resample` makes time-series cleaning a one-liner.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

# Exercise 1 - Clean `titanic`

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
import numpy as np
titanic = load_dataset('titanic')


**Task 1.** How many NaNs are in each column? Print the counts.

In [ ]:
# your code here


**Task 2.** Impute `age` with the median age. Confirm `age` has 0 NaNs after.

In [ ]:
# your code here


**Task 3.** Drop rows where `embarked` is NaN. Report rows lost.

In [ ]:
# your code here


**Task 4.** On the cleaned frame, compute the survival rate by `pclass`.

In [ ]:
# your code here


### Exercise 1 - Solution

# Exercise 1 - Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
import numpy as np
titanic = load_dataset('titanic')


**Task 1.**

In [ ]:
titanic.isna().sum()


**Task 2.**

In [ ]:
titanic = titanic.assign(age=titanic['age'].fillna(titanic['age'].median()))
print('age NaNs:', titanic['age'].isna().sum())


**Task 3.**

In [ ]:
before = len(titanic)
titanic = titanic.dropna(subset=['embarked'])
print('rows lost:', before - len(titanic))


**Task 4.**

In [ ]:
titanic.groupby('pclass')['survived'].mean().round(3)


---

## Exercise 2

# Exercise 2 - GroupBy `tips`

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('tips')


**Task 1.** Add a column `tip_pct = tip / total_bill`.

In [ ]:
# your code here


**Task 2.** Group by `day` and `sex` (use `observed=False`) and compute the mean `tip_pct`.

In [ ]:
# your code here


**Task 3.** Pivot the result of Task 2 so that `day` is the index and `sex` is the columns.

In [ ]:
# your code here


**Task 4.** On which (day, sex) cell is `tip_pct` highest?

In [ ]:
# your code here


### Exercise 2 - Solution

# Exercise 2 - Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('tips')


**Task 1.**

In [ ]:
df = df.assign(tip_pct=df['tip'] / df['total_bill'])
df.head()


**Task 2.**

In [ ]:
g = df.groupby(['day', 'sex'], observed=False)['tip_pct'].mean()
g


**Task 3.**

In [ ]:
pv = g.unstack('sex')
pv


**Task 4.**

In [ ]:
flat = g.reset_index()
top = flat.loc[flat['tip_pct'].idxmax()]
print(top)


---

## Exercise 3

# Exercise 3 - Resample `stock-prices`

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
sp = load_dataset('stock-prices')
print(sp.columns.tolist())
sp.head()


**Task 1.** Parse the date column to datetime and set it as the index. Sort by date.

In [ ]:
# your code here


**Task 2.** Resample the price column from daily to weekly mean.

In [ ]:
# your code here


**Task 3.** Plot the result. (Just `.plot()` - Course 5 covers Matplotlib in depth.)

In [ ]:
# your code here


**Task 4 (stretch).** Resample to monthly mean and plot daily + weekly + monthly on the same axes.

In [ ]:
# your code here


### Exercise 3 - Solution

# Exercise 3 - Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
sp = load_dataset('stock-prices')
DATE  = sp.columns[0]
PRICE = sp.columns[1]


**Task 1.**

In [ ]:
sp[DATE] = pd.to_datetime(sp[DATE])
sp = sp.sort_values(DATE).set_index(DATE)
sp.head()


**Task 2.**

In [ ]:
weekly = sp[PRICE].resample('W').mean()
weekly.head()


**Task 3.**

In [ ]:
ax = weekly.plot(title='AAPL weekly mean')
ax.set_ylabel('USD');


**Task 4.**

In [ ]:
monthly = sp[PRICE].resample('ME').mean()
ax = sp[PRICE].plot(alpha=0.4, label='daily')
weekly.plot(ax=ax, label='weekly')
monthly.plot(ax=ax, label='monthly', lw=2)
ax.legend();
